In [ ]:

import requests
import zipfile
from pathlib import Path

# API officielle data.gouv.fr
API_URL = (
    "https://www.data.gouv.fr/api/1/datasets/"
    "donnees-theoriques-gtfs-et-temps-reel-siri-lite-du-reseau-cts/"
)

# Destination Bronze
destination = Path( "/lakehouse/default/Files/bronze/gtfs_cts/strasbourg"
)
destination.mkdir(parents=True, exist_ok=True)

# 1. Récupérer les métadonnées
response = requests.get(API_URL, timeout=30)
response.raise_for_status()
dataset = response.json()

# 2. Identifier la ressource GTFS
resources = [
    r for r in dataset.get("resources", [])
    if "gtfs" in r.get("title", "").lower()
    and r.get("format", "").lower() in ("zip", "gtfs")
    and r.get("url")
]

if not resources:
    raise RuntimeError("Ressource GTFS introuvable dans l'API.")

resource = max(
    resources,
    key=lambda r: r.get("last_modified") or ""
)

download_url = resource.get("latest") or resource["url"]
file_path = destination / "cts_strasbourg_gtfs.zip"
temp_path = destination / "cts_strasbourg_gtfs.tmp"

print("Ressource :", resource["title"])
print("URL :", download_url)

# 3. Télécharger le ZIP
with requests.get(
    download_url, stream=True, timeout=(20, 120)
) as r:
    r.raise_for_status()
    with open(temp_path, "wb") as f:
        for chunk in r.iter_content(chunk_size=1024 * 1024):
            if chunk:
                f.write(chunk)

# 4. Vérifier le contenu GTFS
with zipfile.ZipFile(temp_path) as z:
    filenames = {Path(n).name for n in z.namelist()}
    required = {"stops.txt", "routes.txt",
                "trips.txt", "stop_times.txt"}
    missing = required - filenames
    if missing:
        raise ValueError(f"Fichiers GTFS manquants : {missing}")

temp_path.replace(file_path)

print("Téléchargement réussi !")
print("Fichier :", file_path)
print("Taille :", round(file_path.stat().st_size / 1048576, 2), "Mo")
print("Fichiers GTFS validés :", len(filenames))


In [ ]:

import pandas as pd
from zipfile import ZipFile
from pathlib import Path

# Fichier GTFS déjà téléchargé
gtfs_path = Path(
    "/lakehouse/default/Files/bronze/"
    "gtfs_cts/strasbourg/cts_strasbourg_gtfs.zip"
)

with ZipFile(gtfs_path) as zip_file:

    files = {
        Path(name).name: name
        for name in zip_file.namelist()
        if not name.endswith("/")
    }

    print("=== FICHIERS GTFS ===")
    for name in sorted(files):
        print("-", name)

    # Lecture des arrêts
    with zip_file.open(files["stops.txt"]) as f:
        stops = pd.read_csv(f, dtype=str)

    # Lecture des lignes
    with zip_file.open(files["routes.txt"]) as f:
        routes = pd.read_csv(f, dtype=str)

print("\n=== STATISTIQUES ===")
print("Nombre de points d'arrêt GTFS :", len(stops))
print("Nombre de lignes GTFS :", len(routes))

print("\n=== EXEMPLES D'ARRÊTS ===")
columns = [
    c for c in ["stop_id", "stop_name", "stop_lat", "stop_lon"]
    if c in stops.columns
]
display(stops[columns].head(10))

print("\n=== EXEMPLES DE LIGNES ===")
columns = [
    c for c in ["route_id", "route_short_name", "route_long_name"]
    if c in routes.columns
]
display(routes[columns].head(10))
